In [1]:
#!pip install adversarial-robustness-toolbox

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [29]:
import numpy as np
import tensorflow as tf
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification  import KerasClassifier
from sklearn.metrics import accuracy_score

In [26]:
tf.compat.v1.disable_eager_execution()

In [30]:
import tensorflow as tf
import json
# download mnist data and split into train and test sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
# reshape data to fit model
X_train, X_test = X_train/255, X_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# create model
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3), activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))


In [31]:
# compile model using accuracy as a measure of model performance
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])

# train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5)

json.dump({'model': model.to_json()}, open("model.json", "w"))
model.save_weights("model_weights.h5")



Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - ETA: 0s - loss: 1.5285 - accuracy: 0.4389

/usr/local/lib/python3.8/dist-packages/keras/engine/training_v1.py:2045: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


50000/50000 [==============================] - 183s 4ms/sample - loss: 1.5285 - accuracy: 0.4389 - val_loss: 1.1532 - val_accuracy: 0.5903
Epoch 2/5
50000/50000 [==============================] - 181s 4ms/sample - loss: 1.1132 - accuracy: 0.6047 - val_loss: 0.9756 - val_accuracy: 0.6564
Epoch 3/5
50000/50000 [==============================] - 181s 4ms/sample - loss: 0.9626 - accuracy: 0.6625 - val_loss: 0.8373 - val_accuracy: 0.7105
Epoch 4/5
50000/50000 [==============================] - 183s 4ms/sample - loss: 0.8787 - accuracy: 0.6918 - val_loss: 0.7950 - val_accuracy: 0.7274
Epoch 5/5
50000/50000 [==============================] - 181s 4ms/sample - loss: 0.8218 - accuracy: 0.7110 - val_loss: 0.7438 - val_accuracy: 0.7385


In [32]:
#Create a KerasClassifier for the model
classifier = KerasClassifier(model=model, clip_values=(0, 1),  use_logits=False)
# Generate adversarial examples
x_test = X_test # your test data
y = y_test # the true labels for the test data
jsma = SaliencyMapMethod(classifier=classifier, theta=1, gamma=0.1)
#x_test_adv = jsma.generate(x=x_test, y=y)
x_test_adv = jsma.generate(x_test)

/usr/local/lib/python3.8/dist-packages/keras/engine/training_v1.py:2067: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


JSMA:   0%|          | 0/10000 [00:00<?, ?it/s]

In [ ]:
print(f"X_train shape: {x_test_adv.shape}")
print(f"y_train shape: {x_test.shape}")

X_train shape: (10000, 32, 32, 3)
y_train shape: (10000, 32, 32, 3)


In [33]:
preds = np.argmax(model.predict(x_test_adv), axis=1)
acc = np.sum(preds == np.argmax(y_test, axis=1)) / y_test.shape[0]
print('Test accuracy on adversarial examples: {}'.format(acc))

Test accuracy on adversarial examples: 0.043


In [34]:
preds = np.argmax(model.predict(X_test), axis=1)
acc = np.sum(preds == np.argmax(y_test, axis=1)) / y_test.shape[0]
acc

0.7385

In [ ]:
preds = np.argmax(model.predict(X_train), axis=1)
acc = np.sum(preds == np.argmax(y_train, axis=1)) / y_train.shape[0]
acc

0.71134

In [ ]:
x_adv_combine=np.concatenate((X_train, x_test_adv), axis=0)

In [ ]:
y_final=np.concatenate((y_train, y_test), axis=0)

In [ ]:
print(f"X_test shape: {x_adv_combine.shape}")
print(f"y_test shape: {y_train.shape}")
print(f"y_test shape: {x_test_adv.shape}")

X_test shape: (60000, 32, 32, 3)
y_test shape: (50000, 10)
y_test shape: (10000, 32, 32, 3)


In [ ]:
model.fit(x_adv_combine, y_final, validation_data=(X_test, y_test), epochs=5)

Train on 60000 samples, validate on 10000 samples
Epoch 1/5
60000/60000 [==============================] - 126s 2ms/sample - loss: 0.7974 - accuracy: 0.7244 - val_loss: 0.9347 - val_accuracy: 0.6805
Epoch 2/5
60000/60000 [==============================] - 123s 2ms/sample - loss: 0.7518 - accuracy: 0.7432 - val_loss: 0.8786 - val_accuracy: 0.7062
Epoch 3/5
60000/60000 [==============================] - 125s 2ms/sample - loss: 0.7144 - accuracy: 0.7555 - val_loss: 0.8424 - val_accuracy: 0.7205
Epoch 4/5
60000/60000 [==============================] - 126s 2ms/sample - loss: 0.6907 - accuracy: 0.7631 - val_loss: 0.8181 - val_accuracy: 0.7302
Epoch 5/5
60000/60000 [==============================] - 126s 2ms/sample - loss: 0.6642 - accuracy: 0.7703 - val_loss: 0.8166 - val_accuracy: 0.7336


In [ ]:
model.fit(X_train, y_train, validation_data=(x_test_adv, y_test), epochs=5)

Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - 107s 2ms/sample - loss: 0.6921 - accuracy: 0.7603 - val_loss: 0.3918 - val_accuracy: 0.8799
Epoch 2/5
50000/50000 [==============================] - 106s 2ms/sample - loss: 0.6611 - accuracy: 0.7692 - val_loss: 0.4686 - val_accuracy: 0.8346
Epoch 3/5
50000/50000 [==============================] - 113s 2ms/sample - loss: 0.6383 - accuracy: 0.7768 - val_loss: 0.4519 - val_accuracy: 0.8423
Epoch 4/5
50000/50000 [==============================] - 106s 2ms/sample - loss: 0.6139 - accuracy: 0.7854 - val_loss: 0.4701 - val_accuracy: 0.8346
Epoch 5/5
50000/50000 [==============================] - 104s 2ms/sample - loss: 0.5940 - accuracy: 0.7911 - val_loss: 0.4780 - val_accuracy: 0.8328


In [ ]:
y_pred_original = model.predict(X_test)
y_pred_original_cat = tf.keras.utils.to_categorical(y_pred_original)

y_test

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], dtype=float32)

In [ ]:
preds_adv = model.predict(x_test_adv)
y_test_adv = tf.keras.utils.to_categorical(preds_adv)

/usr/local/lib/python3.8/dist-packages/keras/engine/training_v1.py:2067: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


In [ ]:
y_pred= model.predict(X_test)

In [ ]:
y_pred

array([[9.52328742e-03, 1.80159323e-02, 1.35985750e-03, ...,
        7.24555648e-05, 4.53779154e-04, 6.40172220e-05],
       [2.79411320e-02, 7.69811645e-02, 4.88494152e-05, ...,
        6.14187599e-08, 8.93988550e-01, 1.03662896e-03],
       [9.01613683e-02, 5.45879714e-02, 6.39276579e-03, ...,
        2.18598591e-03, 7.78007686e-01, 6.02982640e-02],
       ...,
       [7.19824748e-05, 5.35295271e-07, 2.86005456e-02, ...,
        4.11013484e-01, 2.38132361e-05, 1.17627525e-04],
       [1.38369243e-04, 9.99653935e-01, 6.03538547e-06, ...,
        2.36969754e-06, 5.60709168e-08, 1.48636366e-06],
       [6.53084058e-08, 6.11674841e-06, 1.29982072e-06, ...,
        9.99251127e-01, 6.70356641e-08, 1.59282649e-07]], dtype=float32)